# 01 云数据仓库 (Cloud Data Warehouses)

**课程模块**: 云 & 基础设施 — Senior Data Engineer 面试备考

## 本节覆盖考点

| 平台 | 考点 | 频率 |
|------|------|------|
| Snowflake | Virtual Warehouse / Clustering Key | 高频 |
| Snowflake | Time Travel & Zero-Copy Clone | 重要 |
| BigQuery | Slot / Reservation / BI Engine | 高频 |
| BigQuery | Partitioning & Clustering 选择 | 高频 |
| Redshift | Distribution Style / Sort Key | 重要 |
| Databricks | Photon / Delta Engine | 重要 |

---

> **学习方法**: 先读概念（中文解释），再看代码示例，最后做练习。
> 由于云 SDK 未必安装，代码示例使用纯 Python 演示核心逻辑与配置结构。

---
# Part 1: Snowflake

## 1.1 Virtual Warehouse — 计算资源单元

**核心概念**

Snowflake 的计算与存储完全分离。**Virtual Warehouse (VW)** 是计算层的核心概念：

- 本质是一组 MPP 计算节点（EC2 集群），按 T-Shirt size 计费
- **Auto-Suspend**: 闲置 N 秒后自动暂停，停止计费
- **Auto-Resume**: 有查询到来时自动唤醒（通常 < 5秒）
- **Multi-Cluster Warehouse**: 并发量大时自动扩展为多个同等 VW（横向扩展），避免查询排队

```
Snowflake 架构示意图:

  ┌─────────────────────────────────────────────┐
  │           Cloud Services Layer              │
  │  (Auth / Query Planning / Metadata / TT)    │
  └──────────────┬──────────────────────────────┘
                 │
  ┌──────────────▼──────────────────────────────┐
  │          Query Processing Layer             │
  │  ┌──────────┐  ┌──────────┐  ┌──────────┐  │
  │  │ VW: ETL  │  │ VW: BI   │  │ VW: DS   │  │
  │  │  (XL)    │  │  (M)     │  │  (L)     │  │
  │  └──────────┘  └──────────┘  └──────────┘  │
  └──────────────┬──────────────────────────────┘
                 │  (共享存储层)
  ┌──────────────▼──────────────────────────────┐
  │           Storage Layer (S3/Blob/GCS)       │
  │         Columnar Micro-Partitions            │
  └─────────────────────────────────────────────┘
```

**Warehouse Size 与费用**

| Size | Credits/Hour | 适用场景 |
|------|-------------|----------|
| XS   | 1           | 轻量查询、开发测试 |
| S    | 2           | 小型 ETL |
| M    | 4           | 常规 BI 查询 |
| L    | 8           | 大型 ETL / 复杂分析 |
| XL   | 16          | 重型批处理 |
| 2XL  | 32          | 极大数据集 |

> **面试要点**: 纵向扩展（改 Size）加速单个查询；横向扩展（Multi-Cluster）解决并发排队问题。两者解决的问题不同！

In [ ]:
# Snowflake Virtual Warehouse 成本计算示例
# (概念演示，无需实际 Snowflake 连接)

def snowflake_cost_calculator(
    size: str,
    hours_per_day: float,
    days_per_month: int = 22,
    credit_price_usd: float = 3.0,
    multi_cluster_max: int = 1
) -> dict:
    """Calculate Snowflake Virtual Warehouse monthly cost estimate."""
    credits_per_hour = {
        'XS': 1, 'S': 2, 'M': 4, 'L': 8, 'XL': 16, '2XL': 32, '3XL': 64, '4XL': 128
    }
    if size not in credits_per_hour:
        raise ValueError(f"Unknown size: {size}. Choose from {list(credits_per_hour.keys())}")

    credits_per_hour_single = credits_per_hour[size]
    total_credits_per_hour = credits_per_hour_single * multi_cluster_max
    daily_credits = total_credits_per_hour * hours_per_day
    monthly_credits = daily_credits * days_per_month
    monthly_cost = monthly_credits * credit_price_usd

    return {
        'warehouse_size': size,
        'credits_per_hour': credits_per_hour_single,
        'multi_cluster_max': multi_cluster_max,
        'effective_credits_per_hour': total_credits_per_hour,
        'hours_active_per_day': hours_per_day,
        'working_days_per_month': days_per_month,
        'monthly_credits': round(monthly_credits, 2),
        'monthly_cost_usd': round(monthly_cost, 2),
        'note': 'Auto-suspend saves cost when idle; billed per second (min 60s)'
    }


# 场景1: BI 报表仓库，M size，每天活跃8小时，单集群
bi_wh = snowflake_cost_calculator('M', hours_per_day=8, multi_cluster_max=1)
print("=== BI Warehouse (M, single-cluster) ===")
for k, v in bi_wh.items():
    print(f"  {k}: {v}")

print()

# 场景2: ETL 仓库，XL size，每天活跃4小时，无需多集群
etl_wh = snowflake_cost_calculator('XL', hours_per_day=4, multi_cluster_max=1)
print("=== ETL Warehouse (XL, single-cluster) ===")
for k, v in etl_wh.items():
    print(f"  {k}: {v}")

print()

# 场景3: 高并发 BI，M size，多集群最多3个
mc_wh = snowflake_cost_calculator('M', hours_per_day=8, multi_cluster_max=3)
print("=== High-Concurrency BI Warehouse (M, multi-cluster max=3) ===")
for k, v in mc_wh.items():
    print(f"  {k}: {v}")
print("NOTE: Multi-cluster max=3 means UP TO 3 clusters; actual usage depends on concurrency.")

## 1.2 Warehouse DDL 配置示例

以下是 Snowflake SQL DDL 的结构示例（以 Python 字符串展示，便于理解配置选项）：

In [ ]:
# Snowflake DDL 配置示例（字符串展示，无需连接）

create_warehouse_sql = """
-- 创建一个 ETL 专用 Warehouse
CREATE WAREHOUSE etl_wh
    WAREHOUSE_SIZE = 'XLARGE'         -- 计算节点规模
    AUTO_SUSPEND   = 120              -- 空闲 120 秒后暂停（节省费用）
    AUTO_RESUME    = TRUE             -- 有查询时自动唤醒
    MIN_CLUSTER_COUNT = 1
    MAX_CLUSTER_COUNT = 3            -- Multi-Cluster: 并发高时自动扩展到 3 个集群
    SCALING_POLICY = 'ECONOMY'       -- ECONOMY(节省) vs STANDARD(快速响应)
    INITIALLY_SUSPENDED = TRUE       -- 创建后立即暂停，不立刻计费
    COMMENT = 'ETL Pipeline Warehouse - created by data engineering team';
"""

alter_warehouse_sql = """
-- 临时扩容处理月末大批量 ETL
ALTER WAREHOUSE etl_wh SET WAREHOUSE_SIZE = '2XLARGE';

-- 处理完成后缩容回常规大小
ALTER WAREHOUSE etl_wh SET WAREHOUSE_SIZE = 'XLARGE';
"""

monitor_warehouse_sql = """
-- 查看 Warehouse 使用情况（信用额度消耗）
SELECT
    warehouse_name,
    SUM(credits_used)       AS total_credits,
    SUM(credits_used) * 3  AS estimated_cost_usd,
    COUNT(DISTINCT query_id) AS query_count,
    AVG(execution_time) / 1000 AS avg_exec_sec
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -30, CURRENT_TIMESTAMP)
GROUP BY warehouse_name
ORDER BY total_credits DESC;
"""

print("=== CREATE WAREHOUSE ===")
print(create_warehouse_sql)
print("=== ALTER WAREHOUSE (resize) ===")
print(alter_warehouse_sql)
print("=== MONITOR WAREHOUSE USAGE ===")
print(monitor_warehouse_sql)

## 1.3 Clustering Key — 微分区剪枝优化

**Micro-Partition 原理**

Snowflake 自动将数据分为 **Micro-Partitions**（通常 50~500 MB 压缩数据），每个分区记录每列的 min/max 值。查询时，Query Planner 通过这些元数据跳过不相关分区（Partition Pruning）。

**Clustering Key 解决什么问题？**

当数据不按查询过滤条件的自然顺序写入时（如 ORDER BY event_date 写入，但查询经常 WHERE region = 'US'），微分区的 min/max 区间重叠严重，剪枝效果差。

```
无 Clustering Key (region 列分散在所有分区):

  Partition 1: region=[US,EU,APAC], event_date=[2024-01-01,2024-01-02]
  Partition 2: region=[US,EU,APAC], event_date=[2024-01-03,2024-01-04]
  Partition 3: region=[US,EU,APAC], event_date=[2024-01-05,2024-01-06]
  → WHERE region='US' 必须扫描所有分区 (3/3 partitions scanned)

有 CLUSTER BY (region, event_date):

  Partition 1: region=[APAC,APAC], event_date=[2024-01-01,2024-06-30]
  Partition 2: region=[EU,EU],     event_date=[2024-01-01,2024-06-30]
  Partition 3: region=[US,US],     event_date=[2024-01-01,2024-06-30]
  → WHERE region='US' 只扫描 1 个分区 (1/3 partitions scanned)
```

**何时使用 Clustering Key？**

- 表非常大（> 数百 GB）
- 查询频繁按某几列过滤（高基数 + 高频过滤）
- 数据不是按该列自然有序写入的
- 注意：Clustering Key 有维护成本（自动重聚类消耗 Credits）

**Clustering Key 选择决策树**

```
查询频繁按某列 WHERE 过滤?
  ├── 否 → 不需要 Clustering Key
  └── 是 → 表 > 1TB？
        ├── 否 → Automatic Clustering 通常足够
        └── 是 → 列基数如何？
              ├── 低基数 (如 region=5个值) → 适合作为 Clustering Key 第一列
              ├── 高基数 (如 user_id=亿级) → 考虑 DATE_TRUNC('month', ts) 降基
              └── 多列过滤 → 按选择性从低到高排列: (region, date, product_id)
```

In [ ]:
# Snowflake Clustering Key 配置示例与剪枝效率模拟

clustering_key_examples = {
    'sales_events': {
        'sql': 'CREATE TABLE sales_events (event_ts TIMESTAMP, region VARCHAR, product_id INT, amount DECIMAL) CLUSTER BY (region, DATE_TRUNC(\'month\', event_ts));',
        'rationale': '查询常见模式: WHERE region=? AND event_ts BETWEEN ? AND ?',
        'first_col': 'region (低基数~50个值)',
        'second_col': 'month(event_ts) 降基后约240个值'
    },
    'user_activity': {
        'sql': 'CREATE TABLE user_activity (activity_date DATE, user_id BIGINT, action_type VARCHAR) CLUSTER BY (activity_date);',
        'rationale': '查询常见模式: WHERE activity_date >= ? (时间范围扫描)',
        'first_col': 'activity_date (数据已按时间写入，效果显著)'
    }
}

# 微分区剪枝效率模拟
import random

def simulate_partition_pruning(num_partitions: int, num_distinct_values: int,
                                 clustered: bool, query_value: str) -> dict:
    """
    Simulate partition pruning efficiency.
    Without clustering: each partition contains a mix of all values.
    With clustering: partitions are sorted by the cluster key.
    """
    random.seed(42)
    values = [f"val_{i}" for i in range(num_distinct_values)]

    if clustered:
        # Clustered: values are co-located → partitions have narrow value ranges
        # Each partition holds ~(num_partitions / num_distinct_values) partitions per value
        partitions_per_value = max(1, num_partitions // num_distinct_values)
        partitions_scanned = partitions_per_value
    else:
        # Unclustered: each partition contains all values → full scan needed
        partitions_scanned = num_partitions

    pruning_ratio = 1 - (partitions_scanned / num_partitions)
    return {
        'total_partitions': num_partitions,
        'partitions_scanned': partitions_scanned,
        'partitions_skipped': num_partitions - partitions_scanned,
        'pruning_efficiency': f"{pruning_ratio:.1%}",
        'clustered': clustered
    }

print("=== Partition Pruning Simulation: WHERE region = 'US' ===")
print("  Scenario: 1000 partitions, 20 distinct regions")
print()

result_no_cluster = simulate_partition_pruning(1000, 20, clustered=False, query_value='US')
print("Without Clustering Key:")
for k, v in result_no_cluster.items():
    print(f"  {k}: {v}")

print()

result_clustered = simulate_partition_pruning(1000, 20, clustered=True, query_value='US')
print("With CLUSTER BY (region):")
for k, v in result_clustered.items():
    print(f"  {k}: {v}")

print()
print("=== DDL Examples ===")
for table, info in clustering_key_examples.items():
    print(f"\n-- Table: {table}")
    print(f"-- Rationale: {info['rationale']}")
    print(info['sql'])

## 1.4 Time Travel & Zero-Copy Clone

### Time Travel — 时间旅行

Snowflake 在存储层保留历史版本的微分区，允许查询或恢复过去某时刻的数据状态。

- **Standard Edition**: 保留 1 天
- **Enterprise Edition**: 最长保留 90 天
- 配置在 Database / Schema / Table 级别（`DATA_RETENTION_TIME_IN_DAYS`）
- 过期后进入 **Fail-Safe** 期（7天，仅 Snowflake 内部可恢复，不可用户操作）

**查询语法**:
```sql
-- 查询某时间点的历史数据
SELECT * FROM orders AT (TIMESTAMP => '2024-03-01 08:00:00'::TIMESTAMP_LTZ);

-- 查询某查询执行前的数据（用于对比）
SELECT * FROM orders BEFORE (STATEMENT => '019b0345-0504-9dea-0000-0001fd5e8001');

-- 恢复误删的行到新表
CREATE TABLE orders_recovered AS
SELECT * FROM orders AT (OFFSET => -3600);  -- 1小时前

-- 恢复被 DROP 的表
UNDROP TABLE orders;
```

### Zero-Copy Clone — 零拷贝克隆

克隆操作**不复制实际数据**，只复制元数据指针（指向相同的微分区）。写入新数据时才产生存储分叉（Copy-on-Write）。

```
Zero-Copy Clone 示意:

  PROD Table (orders)
  ┌─────────────────────────────────────────┐
  │ Micro-Partition A │ B │ C │ D │ E │ F  │
  └────────┬──────────┴───┴───┴───┴───┴────┘
           │ 元数据指针克隆（瞬时完成，接近零费用）
           ▼
  DEV Table (orders_dev_clone)
  ┌─────────────────────────────────────────┐
  │ Micro-Partition A │ B │ C │ D │ E │ F  │  ← 共享相同物理文件
  └─────────────────────────────────────────┘

  开发者修改 DEV 表后:
  DEV Table
  ┌────────────────────────────────────────────────┐
  │ Micro-Partition A │ B │ C │ D* │ E │ F │ G(新) │
  └────────────────────────────────────────────────┘
                                  ↑
                         只有新写入的 D* 和 G 产生额外存储费用
```

**典型使用场景**:
1. 在生产数据上测试 ETL 逻辑，不影响生产
2. 为数据科学团队提供生产数据快照
3. 部署前备份（克隆后执行 DDL 变更，出错可切回）
4. 多租户环境的数据隔离

In [ ]:
# Time Travel & Zero-Copy Clone SQL 示例（字符串展示）

time_travel_examples = """
-- ============================================================
-- TIME TRAVEL: 3 种引用方式
-- ============================================================

-- 1. 按时间戳 (AT)
SELECT COUNT(*) FROM orders
    AT (TIMESTAMP => DATEADD('hour', -2, CURRENT_TIMESTAMP));

-- 2. 按时间偏移（秒） (OFFSET)
SELECT * FROM orders
    AT (OFFSET => -7200);  -- 7200秒 = 2小时前

-- 3. 按 Query ID (在某个操作执行之前)
SELECT * FROM orders
    BEFORE (STATEMENT => '019b0345-0504-9dea-0000-0001fd5e8001');

-- 恢复: 用历史数据覆盖当前表
CREATE OR REPLACE TABLE orders AS
    SELECT * FROM orders AT (OFFSET => -3600);

-- 恢复被 DROP 的对象
UNDROP TABLE orders;
UNDROP SCHEMA analytics;
UNDROP DATABASE prod_db;

-- 配置 Time Travel 保留期
ALTER TABLE large_events_table SET DATA_RETENTION_TIME_IN_DAYS = 1;  -- 节省存储
ALTER TABLE orders SET DATA_RETENTION_TIME_IN_DAYS = 30;             -- 重要表保留30天
"""

zero_copy_clone_examples = """
-- ============================================================
-- ZERO-COPY CLONE: 各级别克隆
-- ============================================================

-- 克隆表（最常用）
CREATE TABLE orders_dev_20240301 CLONE orders;

-- 克隆历史版本的表（结合 Time Travel）
CREATE TABLE orders_before_migration CLONE orders
    BEFORE (STATEMENT => '019b0345-0504-9dea-0000-0001fd5e8001');

-- 克隆 Schema（克隆其中所有表）
CREATE SCHEMA prod_analytics_clone CLONE prod_analytics;

-- 克隆整个 Database（完整数据库级别隔离）
CREATE DATABASE prod_db_clone CLONE prod_db;

-- 典型工作流: 安全部署 DDL 变更
-- Step 1: 克隆生产表
CREATE TABLE orders_backup CLONE orders;
-- Step 2: 在生产表上执行 DDL 变更
ALTER TABLE orders ADD COLUMN discount_rate DECIMAL(5,4);
-- Step 3: 如果出错，从备份恢复
-- CREATE OR REPLACE TABLE orders CLONE orders_backup;
"""

print("=== TIME TRAVEL EXAMPLES ===")
print(time_travel_examples)
print("\n=== ZERO-COPY CLONE EXAMPLES ===")
print(zero_copy_clone_examples)

# 演示存储节省计算
def clone_storage_savings(table_size_gb: float, change_rate_percent: float = 10.0) -> dict:
    """Estimate storage cost for zero-copy clone vs full copy."""
    full_copy_gb = table_size_gb
    clone_overhead_gb = table_size_gb * (change_rate_percent / 100)
    saving_gb = full_copy_gb - clone_overhead_gb
    # Snowflake storage ~$23/TB/month
    storage_price_per_gb_month = 23.0 / 1024
    monthly_saving_usd = saving_gb * storage_price_per_gb_month
    return {
        'original_table_gb': table_size_gb,
        'full_copy_would_cost_gb': full_copy_gb,
        'clone_actual_overhead_gb': round(clone_overhead_gb, 2),
        'storage_saved_gb': round(saving_gb, 2),
        'monthly_saving_usd': round(monthly_saving_usd, 2)
    }

print("\n=== Zero-Copy Clone Storage Savings ===")
result = clone_storage_savings(table_size_gb=5000, change_rate_percent=5)
for k, v in result.items():
    print(f"  {k}: {v}")

---
# Part 2: BigQuery

## 2.1 Slot / Reservation / BI Engine

### BigQuery 计算模型

BigQuery 的计费模型有两种：

| 模式 | 计费方式 | 适用场景 |
|------|---------|----------|
| On-demand | 按扫描字节数（$5/TB） | 不规律查询、探索性分析 |
| Capacity (Reservations) | 按 Slot 数量（按小时/月/年） | 稳定工作负载、成本可预测 |

### Slot 是什么？

**Slot** 是 BigQuery 的计算资源单元（类似于一个 CPU 虚拟线程）。BigQuery 是 Dremel 架构，查询被分解为多个 stage，每个 stage 并行跨多个 slot 执行。

```
BigQuery 计算架构:

  Query: SELECT region, SUM(revenue) FROM sales GROUP BY region
         │
         ▼
  ┌─────────────────────────────────────────────┐
  │               Query Plan                   │
  │  Stage 1: Read + Filter (scan workers)     │
  │  Stage 2: Partial Aggregation (shuffle)    │
  │  Stage 3: Final Aggregation + Output       │
  └─────────────────────────────────────────────┘
         │ 分配给可用 Slots
         ▼
  ┌──────────────────────────────────────────────┐
  │ Reservation: analytics_team (500 slots)      │
  │  ┌─────────┐ ┌─────────┐ ┌─────────┐        │
  │  │Slot 001 │ │Slot 002 │ │Slot 003 │  ...   │
  │  └─────────┘ └─────────┘ └─────────┘        │
  └──────────────────────────────────────────────┘
```

### Reservation 层级

```
Admin Project
└── Commitment (购买的 Slot 容量: 1000 slots, 1-year)
    ├── Reservation: etl_jobs (600 slots)
    │   └── Assignment: project-A → etl_jobs reservation
    └── Reservation: bi_queries (400 slots)
        └── Assignment: project-B → bi_queries reservation
```

- **Commitment**: 承诺购买（Flex=60秒, Standard=1年, Enterprise=3年），折扣递增
- **Reservation**: Slot 池，可以设置 `ignore_idle_slots=false` 借用空闲 Slot
- **Assignment**: 将项目/文件夹分配到某个 Reservation

### BI Engine

**BI Engine** 是 BigQuery 内置的内存加速层，专为 BI 工具（Looker, Data Studio）设计：

- 将热数据缓存到内存中（按 GB 计费，~$0.04/GB/hour）
- 对支持的查询模式（简单聚合、过滤）实现亚秒响应
- 与 Reservation 配合使用效果最佳
- 不支持复杂 JOIN、ARRAY/STRUCT 等操作

In [ ]:
# BigQuery 成本模型对比计算

def bq_ondemand_cost(tb_scanned_per_month: float, price_per_tb: float = 5.0) -> dict:
    """Calculate BigQuery on-demand query cost."""
    return {
        'model': 'On-Demand',
        'tb_scanned_per_month': tb_scanned_per_month,
        'price_per_tb_usd': price_per_tb,
        'monthly_cost_usd': round(tb_scanned_per_month * price_per_tb, 2),
        'notes': 'First 1TB/month free. Cost varies with query volume.'
    }


def bq_reservation_cost(
    slots: int,
    commitment_type: str = 'flex',
    hours_per_month: int = 730
) -> dict:
    """Calculate BigQuery reservation (capacity) cost."""
    # Approximate pricing (us-central1, 2024)
    price_per_slot_hour = {
        'flex':       0.04,    # per slot-hour
        'standard':   0.02,    # 1-year commitment, per slot-hour equivalent
        'enterprise': 0.015,   # 3-year commitment, per slot-hour equivalent
    }
    if commitment_type not in price_per_slot_hour:
        raise ValueError(f"commitment_type must be one of {list(price_per_slot_hour.keys())}")

    rate = price_per_slot_hour[commitment_type]
    monthly_cost = slots * rate * hours_per_month
    return {
        'model': f'Reservation ({commitment_type})',
        'slots': slots,
        'commitment_type': commitment_type,
        'price_per_slot_hour_usd': rate,
        'hours_per_month': hours_per_month,
        'monthly_cost_usd': round(monthly_cost, 2),
        'notes': 'Fixed cost regardless of data scanned. Good for predictable workloads.'
    }


print("=" * 60)
print("BigQuery Cost Model Comparison")
print("=" * 60)

# On-demand: scan 500TB/month
od = bq_ondemand_cost(tb_scanned_per_month=500)
print("\nOn-Demand (500 TB/month scanned):")
for k, v in od.items():
    print(f"  {k}: {v}")

# Flex Reservation: 500 slots
flex = bq_reservation_cost(slots=500, commitment_type='flex')
print("\nFlex Reservation (500 slots):")
for k, v in flex.items():
    print(f"  {k}: {v}")

# Standard (1-year) Reservation: 500 slots
std = bq_reservation_cost(slots=500, commitment_type='standard')
print("\nStandard 1-Year Reservation (500 slots):")
for k, v in std.items():
    print(f"  {k}: {v}")

print("\n" + "=" * 60)
print("Breakeven Analysis:")
# At what scan volume does reservation become cheaper than on-demand?
flex_monthly = flex['monthly_cost_usd']
breakeven_tb = flex_monthly / 5.0
print(f"  Flex (500 slots) costs ${flex_monthly}/month")
print(f"  Breakeven vs On-Demand: {breakeven_tb:.0f} TB/month scanned")
print(f"  → If you scan > {breakeven_tb:.0f} TB/month, Flex Reservation is cheaper")

std_monthly = std['monthly_cost_usd']
breakeven_tb_std = std_monthly / 5.0
print(f"\n  Standard (500 slots) costs ${std_monthly}/month (annualized)")
print(f"  Breakeven vs On-Demand: {breakeven_tb_std:.0f} TB/month scanned")

## 2.2 BigQuery Partitioning & Clustering 选择

### Partitioning (分区)

BigQuery 将数据物理分割为独立分区，查询时只读取匹配的分区（**分区消除**），直接减少计费字节。

| 分区类型 | 语法 | 适用场景 |
|---------|------|----------|
| 按摄入时间 | `PARTITION BY _PARTITIONTIME` | 无时间列但需按写入时间切分 |
| 按时间列 | `PARTITION BY DATE(event_ts)` | 有业务时间列（推荐） |
| 按整数范围 | `PARTITION BY RANGE_BUCKET(user_id, GENERATE_ARRAY(0,10000,100))` | 非时间维度分区 |

### Clustering (聚簇)

在分区内部，按指定列排序存储。查询时通过 Block 元数据跳过不匹配的 Block（**Block 剪枝**）。

- 最多支持 4 个聚簇列
- 列的顺序决定剪枝效果：过滤条件匹配前缀列才能剪枝
- 聚簇不减少计费字节（分区才减费用），但显著加快查询速度

### 选择决策树

```
查询是否有时间范围过滤 (WHERE date BETWEEN ...)?
  └── 是 → 按时间列分区 (PARTITION BY DATE(event_ts))
           查询是否还有其他高频过滤列?
           └── 是 → 加 CLUSTER BY (region, product_type)
           └── 否 → 仅分区即可
  └── 否 → 数据有自然分组维度?
           └── 是 → 考虑整数范围分区或仅 CLUSTER BY
           └── 否 → 考虑是否需要优化（表可能不够大）

分区 vs 聚簇 核心区别:
  分区: 减少扫描字节 → 直接降低查询费用
  聚簇: 加快查询速度 → 不直接降低费用，但改善用户体验
```

**最佳实践组合**: 分区（减费用） + 聚簇（减延迟）

In [ ]:
# BigQuery Partitioning & Clustering DDL 示例

bq_ddl_examples = {
    'e_commerce_events': """
-- E-commerce 事件表: 按日期分区 + 多列聚簇
CREATE TABLE `project.dataset.e_commerce_events`
(
    event_ts        TIMESTAMP NOT NULL,
    event_date      DATE      NOT NULL,  -- 分区列建议存为 DATE
    user_id         INT64,
    event_type      STRING,              -- 'click', 'view', 'purchase'
    product_id      INT64,
    region          STRING,
    revenue         FLOAT64
)
PARTITION BY event_date
CLUSTER BY region, event_type, product_id
OPTIONS (
    partition_expiration_days = 365,     -- 自动删除超过1年的分区
    require_partition_filter = TRUE      -- 强制查询必须带分区过滤（防止全表扫描）
);
""",
    'sensor_readings': """
-- IoT 传感器数据: 按月分区（数据量大，细粒度分区太多）
CREATE TABLE `project.dataset.sensor_readings`
(
    reading_ts      TIMESTAMP NOT NULL,
    reading_month   DATE,               -- EXTRACT(YEAR/MONTH) 后存储
    sensor_id       STRING,
    factory_id      STRING,
    metric_name     STRING,
    value           FLOAT64
)
PARTITION BY DATE_TRUNC(reading_month, MONTH)
CLUSTER BY factory_id, sensor_id;
""",
    'user_segments': """
-- 用户分段表: 整数范围分区（按 user_id 分桶）
CREATE TABLE `project.dataset.user_segments`
(
    user_id         INT64 NOT NULL,
    segment         STRING,
    updated_at      TIMESTAMP
)
PARTITION BY RANGE_BUCKET(user_id, GENERATE_ARRAY(0, 100000000, 1000000))  -- 100个分区
CLUSTER BY segment;
"""
}

print("BigQuery Partitioning & Clustering DDL Examples")
print("=" * 60)
for table_name, ddl in bq_ddl_examples.items():
    print(f"\n### {table_name} ###")
    print(ddl)

# 分区剪枝收益估算
def bq_partition_savings(total_table_tb: float, query_date_range_days: int,
                          table_retention_days: int = 365) -> dict:
    """Estimate BigQuery cost savings from date partitioning."""
    tb_per_day = total_table_tb / table_retention_days
    tb_scanned_without_partition = total_table_tb
    tb_scanned_with_partition = tb_per_day * query_date_range_days
    cost_saved_per_query = (tb_scanned_without_partition - tb_scanned_with_partition) * 5.0

    return {
        'table_total_tb': total_table_tb,
        'query_scans_days': query_date_range_days,
        'tb_without_partitioning': round(tb_scanned_without_partition, 2),
        'tb_with_partitioning': round(tb_scanned_with_partition, 4),
        'cost_saved_per_query_usd': round(cost_saved_per_query, 2),
        'pruning_ratio': f"{(1 - query_date_range_days/table_retention_days):.1%}"
    }

print("\n" + "=" * 60)
print("Partition Savings Estimate: 10TB table, query last 7 days")
savings = bq_partition_savings(total_table_tb=10, query_date_range_days=7, table_retention_days=365)
for k, v in savings.items():
    print(f"  {k}: {v}")

---
# Part 3: Amazon Redshift

## 3.1 Distribution Style — 数据分布策略

Redshift 是 MPP（大规模并行处理）数据库，数据分布在多个节点上。**Distribution Style** 决定行如何分布到各节点，直接影响查询是否需要网络数据传输（**Data Movement**）。

```
Redshift 集群架构:

  ┌─────────────────────────────────────────────────┐
  │                  Leader Node                    │
  │         (Query Planning & Coordination)         │
  └──────┬──────────────┬───────────────┬───────────┘
         │              │               │
  ┌──────▼──┐    ┌──────▼──┐    ┌──────▼──┐
  │ Compute │    │ Compute │    │ Compute │
  │ Node 1  │    │ Node 2  │    │ Node 3  │
  │ Slice1  │    │ Slice3  │    │ Slice5  │
  │ Slice2  │    │ Slice4  │    │ Slice6  │
  └─────────┘    └─────────┘    └─────────┘
```

### 4 种 Distribution Style 对比

| Style | 原理 | 优点 | 缺点 | 适用场景 |
|-------|------|------|------|----------|
| **EVEN** | 行按 Round-Robin 均匀分发 | 均匀分布，无倾斜 | JOIN 需全量广播/重分布 | 无明显 JOIN 键，数据均匀访问 |
| **KEY** | 按指定列 Hash 分发 | 相同 key 在同一节点，JOIN 本地化 | 可能数据倾斜 | 经常与另一大表 JOIN 的表 |
| **ALL** | 每个节点存完整副本 | 小表 JOIN 零数据移动 | 存储消耗 N 倍，写入慢 | 小维度表（< 几百万行）|
| **AUTO** | 小表用 ALL，大表用 EVEN | 自动优化，无需人工决策 | 控制力弱 | 默认设置，适合大多数场景 |

### KEY Distribution 的数据倾斜问题

```
如果 distkey 列数据不均匀 (如 country='US' 占 80% 数据):

  Node 1: ████████████████████████████████ (US 用户 80%)
  Node 2: ████ (EU 用户 15%)
  Node 3: █ (APAC 用户 5%)

  → Node 1 成为热点，查询速度受限于最慢节点
  → 查看倾斜: SELECT slice, count(*) FROM sales GROUP BY 1 ORDER BY 2 DESC;
```

In [ ]:
# Redshift Distribution Style & Sort Key DDL 示例

redshift_ddl = {
    'fact_orders_key': """
-- 大事实表: KEY 分布（与 customers 表 JOIN 优化）
-- customers 表也使用 customer_id 作为 distkey，实现 Collocated JOIN
CREATE TABLE fact_orders (
    order_id      BIGINT    NOT NULL,
    customer_id   BIGINT    NOT NULL,
    order_date    DATE      NOT NULL,
    product_id    INT,
    amount        DECIMAL(12,2),
    region        VARCHAR(50)
)
DISTKEY (customer_id)           -- 与 customers 表共用 distkey → Collocated JOIN
COMPOUND SORTKEY (order_date, region);  -- 查询常按日期+地区过滤
""",
    'dim_customers_all': """
-- 小维度表: ALL 分布（每个节点都有完整副本）
-- 适合经常与大表 JOIN 的小维度表
CREATE TABLE dim_customers (
    customer_id   BIGINT    NOT NULL,
    customer_name VARCHAR(255),
    email         VARCHAR(255),
    country       VARCHAR(50),
    segment       VARCHAR(50)
)
DISTSTYLE ALL
SORTKEY (customer_id);
""",
    'staging_events_even': """
-- Staging 表: EVEN 分布（无 JOIN，只做 COPY 和转换）
CREATE TABLE staging_events (
    event_id      VARCHAR(64),
    event_ts      TIMESTAMP,
    raw_payload   VARCHAR(65535)
)
DISTSTYLE EVEN;
"""
}

print("=== Redshift Distribution Style DDL Examples ===")
for name, ddl in redshift_ddl.items():
    print(f"\n-- {name} --")
    print(ddl)

# Sort Key 类型对比
sortkey_comparison = """
=== COMPOUND vs INTERLEAVED Sort Key ===

COMPOUND SORTKEY (col1, col2, col3):
  - 数据按 col1 排序，col1 相同的再按 col2 排序，以此类推
  - WHERE col1=? → 高效 (跳过不匹配的 blocks)
  - WHERE col2=? (不带 col1) → 低效 (col2 不在前缀)
  - 适合: 有明确主要过滤列（如 event_date 总是第一过滤条件）

INTERLEAVED SORTKEY (col1, col2, col3):
  - 用 Z-order 曲线交织排序，所有列权重相等
  - WHERE 任意列 → 均能受益
  - 维护成本高（VACUUM REINDEX 耗时长）
  - 适合: 多列过滤且无明显主次之分
  - 注意: ra3 节点上已不推荐使用，性能改进有限

VACUUM & ANALYZE:
  VACUUM SORT ONLY fact_orders;    -- 重新排序，不回收空间
  VACUUM DELETE ONLY fact_orders;  -- 只回收已删除行的空间
  VACUUM FULL fact_orders;         -- 完整 VACUUM（耗时最长）
  ANALYZE fact_orders;             -- 更新表统计信息（影响查询计划）
"""
print(sortkey_comparison)

---
# Part 4: Databricks — Photon & Delta Engine

## 4.1 Photon Engine — 向量化 C++ 执行引擎

**Photon** 是 Databricks 开发的原生向量化执行引擎，用 C++ 编写，替代 JVM-based Spark 执行层。

```
传统 Spark JVM 执行 vs Photon 向量化执行:

  JVM (行处理):
  ┌──────┬──────┬──────┬──────┐
  │ Row1 │ Row2 │ Row3 │ Row4 │  每次处理一行，JVM 对象开销大
  └──────┴──────┴──────┴──────┘

  Photon (列批处理, SIMD 向量化):
  ┌──────────────────────────────────────┐
  │ col_A: [1.0, 2.0, 3.0, 4.0, ...   ] │  一次处理整列的 batch
  │ col_B: [a,   b,   c,   d,   ...   ] │  利用 CPU SIMD 指令集并行计算
  └──────────────────────────────────────┘
```

**Photon 最能发挥效果的场景**:
- 大量数值计算（聚合、数学函数）
- 大表 JOIN（Hash Join）
- 宽表扫描 + 过滤
- SQL 工作负载（比 Python UDF 收益更大）

**Photon 不能加速的场景**:
- Python/R UDF（跨进程调用）
- Streaming（部分支持）
- 某些复杂分析函数

## 4.2 Delta Engine — Delta Lake 优化执行

**Delta Engine** 是 Delta Lake 在 Databricks 上的优化查询层，包含多项针对 Delta 表格式的优化：

| 优化技术 | 描述 | 收益 |
|---------|------|------|
| **Data Skipping** | 利用 Delta Log 中的列统计信息跳过不相关文件 | 减少 IO |
| **Z-Ordering** | 多维空间曲线排序，协同定位相关数据 | 减少文件扫描 |
| **Auto Optimize** | 自动小文件合并（Auto Compaction + Optimized Writes）| 减少小文件问题 |
| **Bloom Filter** | 为高基数列构建 Bloom Filter 索引 | 加速点查 |
| **Liquid Clustering** | 新一代替代 Z-Order 的动态聚簇（DBR 13.3+）| 增量重聚簇 |

```
Delta Lake 文件优化策略选择:

  单列过滤 (WHERE date = '2024-01')?
    └── PARTITION BY date (传统分区，每个分区独立目录)

  多列过滤 (WHERE country='US' AND category='Electronics')?
    └── OPTIMIZE ... ZORDER BY (country, category)  [DBR < 13.3]
    └── CLUSTER BY (country, category)              [Liquid Clustering, DBR >= 13.3]

  点查 (WHERE user_id = 12345)?
    └── Bloom Filter Index ON user_id
```

In [ ]:
# Databricks Delta Engine 配置与优化示例

delta_optimize_examples = """
-- ============================================================
-- Delta Lake 优化命令 (Databricks SQL / PySpark)
-- ============================================================

-- 1. OPTIMIZE: 合并小文件 + Z-Order 排序
OPTIMIZE delta.`/mnt/datalake/events`
    ZORDER BY (country, event_type);

-- 2. Bloom Filter Index: 高基数列点查优化
CREATE BLOOMFILTER INDEX ON TABLE events
    FOR COLUMNS (user_id OPTIONS (fpp=0.1, numItems=50000000));

-- 3. Liquid Clustering (DBR 13.3+): 替代 ZORDER
CREATE TABLE events
    CLUSTER BY (country, event_type)
    AS SELECT * FROM raw_events;

-- 4. Auto Optimize 配置
ALTER TABLE events SET TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',  -- 写入时自动优化文件大小
    'delta.autoOptimize.autoCompact'   = 'true'   -- 自动触发小文件合并
);
"""

pyspark_delta_example = """
# PySpark: 使用 Delta Lake API 进行优化
from delta.tables import DeltaTable

# OPTIMIZE + ZORDER
dt = DeltaTable.forPath(spark, '/mnt/datalake/events')
dt.optimize().executeZOrderBy('country', 'event_type')

# Time Travel 查询（类似 Snowflake）
df_yesterday = spark.read.format('delta') \\
    .option('timestampAsOf', '2024-03-01') \\
    .load('/mnt/datalake/events')

df_version5 = spark.read.format('delta') \\
    .option('versionAsOf', 5) \\
    .load('/mnt/datalake/events')

# RESTORE: 回滚到历史版本
dt.restoreToVersion(5)

# VACUUM: 清理历史文件（保留7天）
dt.vacuum(168)  # 168 hours = 7 days
"""

print("=== Delta Lake Optimize Commands (SQL) ===")
print(delta_optimize_examples)
print("\n=== PySpark Delta API ===")
print(pyspark_delta_example)

# Photon 收益估算（概念模型）
print("\n=== Photon Speedup Estimation (Approximate) ===")
workloads = [
    ('Large table scan + filter',         3.0, 5.0),
    ('Hash join (large tables)',          4.0, 8.0),
    ('GROUP BY aggregation',              2.5, 4.0),
    ('Python UDF',                        1.0, 1.0),  # Photon does not accelerate Python UDFs
    ('Window functions',                  2.0, 3.5),
    ('Complex nested subqueries',         1.5, 2.5),
]

print(f"{'Workload Type':<35} {'Min Speedup':>12} {'Max Speedup':>12}")
print("-" * 62)
for workload, min_speedup, max_speedup in workloads:
    print(f"{workload:<35} {min_speedup:>10.1f}x {max_speedup:>10.1f}x")

---
# Part 5: 平台横向对比

## 5.1 云数仓选型决策矩阵

| 维度 | Snowflake | BigQuery | Redshift | Databricks |
|------|-----------|----------|----------|------------|
| **定价模型** | Credits (计算+存储分开) | 按扫描量 or Slots | 按节点类型 | DBU (Databricks Units) |
| **扩展方式** | 自动 Multi-Cluster | Serverless 自动 | 手动 resize / Elastic | 自动 Autoscaling |
| **最强场景** | 多工作负载并发隔离 | 超大数据集临时查询 | AWS 生态深度集成 | ML/AI + 数据工程统一 |
| **存储格式** | 专有格式 | 专有格式 | 列存专有 | Delta Lake (开放) |
| **Time Travel** | 0-90天（Enterprise）| 7天（TIMESTAMP） | 无原生 | 30天（Delta Log）|
| **云支持** | AWS/Azure/GCP | GCP原生 | AWS原生 | AWS/Azure/GCP |
| **典型客户** | 多租户 SaaS，BI | 初创+互联网 | 金融+零售(AWS) | 数据+AI一体化 |

## 5.2 面试常见问题速答

**Q: Snowflake Clustering Key 和 BigQuery Clustering 有什么区别？**
> Snowflake 的 Clustering Key 作用于微分区级别，通过重写/重排微分区来减少扫描；BQ Clustering 在分区内部排序 Block，通过 Block 元数据跳过。两者原理类似但实现不同。Snowflake 的自动聚类持续在后台维护，BigQuery 的聚类在查询时自动剪枝。

**Q: BigQuery 什么时候用 partition，什么时候用 cluster？**
> Partition 减少**计费字节**（物理分隔，查询不读不相关分区）；Cluster 减少**查询延迟**（分区内排序，block 级跳过）。两者可以同时使用：先 partition 减费用，再 cluster 加速。

**Q: Redshift DISTKEY 选择时如何避免数据倾斜？**
> 选择高基数且均匀分布的列作为 DISTKEY（如 order_id、user_id），避免低基数列（如 country、status）。可以用 `SVV_TABLE_INFO` 查看倾斜比率（skew_rows）。

**Q: 什么是 Photon，与 Apache Spark 的关系？**
> Photon 是 Databricks 开发的 C++ 向量化执行引擎，与 Spark 的逻辑计划层兼容（相同 API），但替换了 JVM 的物理执行层。Photon 通过 SIMD 指令批量处理列数据，对 SQL 工作负载有 2-8x 的加速效果，但 Python UDF 不能被加速。

In [ ]:
# 云数仓平台特性对比（结构化数据）

platforms = {
    'Snowflake': {
        'compute_unit': 'Credit (1 XS = 1 credit/hr)',
        'auto_scale': 'Multi-Cluster Warehouse (横向) + Resize (纵向)',
        'time_travel_days': 90,
        'zero_copy_clone': True,
        'serverless': 'Snowpark Container Services',
        'open_format': False,
        'best_for': 'Multi-workload isolation, SaaS analytics'
    },
    'BigQuery': {
        'compute_unit': 'Slot (1 slot = 1 CPU thread equivalent)',
        'auto_scale': 'Serverless (on-demand) or Autoscaling Reservations',
        'time_travel_days': 7,
        'zero_copy_clone': False,
        'serverless': 'Native serverless (no cluster management)',
        'open_format': False,
        'best_for': 'Ad-hoc analytics on massive datasets, GCP ecosystem'
    },
    'Redshift': {
        'compute_unit': 'Node (ra3, dc2 types)',
        'auto_scale': 'Elastic Resize + Concurrency Scaling',
        'time_travel_days': 0,  # No native Time Travel
        'zero_copy_clone': False,
        'serverless': 'Redshift Serverless (RPUS billing)',
        'open_format': False,
        'best_for': 'AWS ecosystem integration, traditional BI'
    },
    'Databricks': {
        'compute_unit': 'DBU (Databricks Unit, varies by cluster type)',
        'auto_scale': 'Autoscaling clusters + Serverless compute',
        'time_travel_days': 30,  # Delta Lake default retention
        'zero_copy_clone': True,  # Delta Lake shallow clone
        'serverless': 'Serverless SQL Warehouse + Serverless Jobs',
        'open_format': True,  # Delta Lake is open source
        'best_for': 'Unified data + AI/ML platform, open ecosystem'
    }
}

# 打印对比表格
print("Cloud Data Warehouse Platform Comparison")
print("=" * 75)

attributes = ['compute_unit', 'auto_scale', 'time_travel_days', 'zero_copy_clone',
              'serverless', 'open_format', 'best_for']

for attr in attributes:
    print(f"\n{attr.upper().replace('_', ' ')}:")
    for platform, props in platforms.items():
        value = props.get(attr, 'N/A')
        print(f"  {platform:<15}: {value}")

---
# 练习题

## 练习 1: Snowflake Virtual Warehouse 设计

某公司有以下工作负载需求：
- ETL Pipeline：每天凌晨 2-6 点运行，处理 500GB 数据，需要快速完成
- BI 报表：工作时间 9am-6pm，峰值 50 个并发用户，查询较轻量
- 数据科学：不定期，需要运行复杂 ML 特征工程查询，1-2 个用户

**问题**:
1. 应该创建几个 Virtual Warehouse？每个应该选什么 Size？
2. BI 仓库是否需要 Multi-Cluster？如果需要，Min/Max 应该设为多少？
3. 如何配置 Auto-Suspend 以最小化费用？

在下面单元格中写出你的设计和理由：

In [ ]:
# 练习 1 解答区域
# 提示: 考虑每个工作负载的特点（并发 vs 单查询复杂度）

exercise_1_answer = {
    'warehouse_design': [
        # 在这里填写你的设计
        # 示例格式:
        # {'name': 'etl_wh', 'size': '???', 'multi_cluster': False, 'auto_suspend_sec': ???, 'rationale': '???'}
    ]
}

# ---- 参考答案（折叠查看）----
reference_answer_1 = {
    'warehouse_design': [
        {
            'name': 'etl_wh',
            'size': 'XL',
            'multi_cluster_min': 1,
            'multi_cluster_max': 1,
            'auto_suspend_sec': 60,
            'rationale': '单查询性能敏感，纵向扩展。ETL 窗口结束后立即挂起节省费用。'
        },
        {
            'name': 'bi_wh',
            'size': 'M',
            'multi_cluster_min': 1,
            'multi_cluster_max': 3,
            'auto_suspend_sec': 300,
            'rationale': '50并发用户需要横向扩展解决排队问题。BI 查询不复杂，M 够用。5分钟挂起平衡响应速度和费用。'
        },
        {
            'name': 'ds_wh',
            'size': 'L',
            'multi_cluster_min': 1,
            'multi_cluster_max': 1,
            'auto_suspend_sec': 120,
            'rationale': '复杂特征工程需要较大计算，但用户少，不需要多集群。'
        }
    ]
}

print("=== 练习 1 参考答案 ===")
for wh in reference_answer_1['warehouse_design']:
    print(f"\nWarehouse: {wh['name']}")
    for k, v in wh.items():
        if k != 'name':
            print(f"  {k}: {v}")

## 练习 2: BigQuery Partitioning & Clustering 选择

你有一张 `user_events` 表，结构如下：
```
user_id     INT64
event_ts    TIMESTAMP
country     STRING    -- 200个不同值
event_type  STRING    -- 50个不同值
device      STRING    -- 5个不同值
revenue     FLOAT64
```

表大小：50TB，每天新增 100GB。

最常见查询模式：
```sql
-- 查询1: 按日期范围 + 国家过滤
WHERE event_ts BETWEEN '2024-01-01' AND '2024-01-31' AND country = 'US'

-- 查询2: 按日期范围 + 事件类型
WHERE DATE(event_ts) = '2024-01-15' AND event_type = 'purchase'

-- 查询3: 只按日期
WHERE DATE(event_ts) >= '2024-01-01'
```

**问题**: 写出最优的 DDL，包含分区和聚簇策略，并解释为什么这样设计。

In [ ]:
# 练习 2 解答区域
your_ddl = """
-- 在这里写你的 DDL
CREATE TABLE `project.dataset.user_events` (
    -- ...
)
PARTITION BY ???
CLUSTER BY ???;
"""

# ---- 参考答案 ----
reference_ddl = """
CREATE TABLE `project.dataset.user_events`
(
    user_id     INT64,
    event_ts    TIMESTAMP NOT NULL,
    event_date  DATE NOT NULL,        -- 物化日期列，避免重复计算
    country     STRING,
    event_type  STRING,
    device      STRING,
    revenue     FLOAT64
)
PARTITION BY event_date               -- 按日期分区: 所有3个查询都带日期过滤 → 减少计费字节
CLUSTER BY country, event_type        -- 聚簇: 查询1用country，查询2用event_type，前缀匹配
OPTIONS (
    partition_expiration_days = 365,
    require_partition_filter  = TRUE  -- 强制分区过滤，防止意外全表扫描
);

-- 设计理由:
-- 1. PARTITION BY event_date: 三个查询模式均有日期过滤，分区消除直接减少扫描字节（降费用）
-- 2. CLUSTER BY (country, event_type): 查询1用country(第一列完全剪枝), 查询2用event_type
--    注意: country 基数(200)> event_type(50)，但 country 出现在更多查询中 → 排第一
-- 3. device 不加入 CLUSTER: 基数太低(5个值)，剪枝效果差，浪费聚簇列名额
-- 4. require_partition_filter: 50TB大表防止运维失误产生巨额账单
"""

print("=== 练习 2 参考答案 ===")
print(reference_ddl)

## 练习 3: Redshift Distribution Style 选择

给定以下表格和查询，为每个表选择最优的 Distribution Style：

```sql
-- 表结构
fact_sales      (sale_id, customer_id, product_id, date, amount)  -- 10亿行
dim_customers   (customer_id, name, country, segment)             -- 500万行
dim_products    (product_id, name, category, price)               -- 5万行
dim_date        (date_id, date, week, month, quarter, year)        -- 3650行

-- 最频繁的查询
SELECT c.country, p.category, SUM(s.amount)
FROM fact_sales s
    JOIN dim_customers c ON s.customer_id = c.customer_id   -- 大表JOIN
    JOIN dim_products  p ON s.product_id  = p.product_id    -- 中小表JOIN
    JOIN dim_date      d ON s.date        = d.date_id        -- 极小表JOIN
WHERE d.year = 2024
GROUP BY c.country, p.category;
```

**问题**: 为每张表选择 Distribution Style 并说明理由。

In [ ]:
# 练习 3 参考答案

reference_answer_3 = [
    {
        'table': 'fact_sales',
        'size': '10亿行 (~数百GB)',
        'distkey': 'DISTKEY(customer_id)',
        'rationale': (
            'customer_id 基数高，分布均匀，不会倾斜。'
            '与 dim_customers 使用相同 distkey → Collocated JOIN (无数据移动)。'
            '是所有 JOIN 中数据最大的，优先保证大JOIN本地化。'
        )
    },
    {
        'table': 'dim_customers',
        'size': '500万行 (~几十GB)',
        'distkey': 'DISTKEY(customer_id)',
        'rationale': (
            '与 fact_sales 共用 customer_id 作为 distkey，实现 Collocated JOIN。'
            '500万行不够小（不适合 ALL），也不应用 EVEN（会破坏 JOIN 本地化）。'
        )
    },
    {
        'table': 'dim_products',
        'size': '5万行 (很小)',
        'distkey': 'DISTSTYLE ALL',
        'rationale': (
            '5万行非常小，ALL 分布复制到所有节点的存储开销可忽略不计。'
            '与 fact_sales 的 product_id JOIN 零数据移动。'
        )
    },
    {
        'table': 'dim_date',
        'size': '3650行 (极小)',
        'distkey': 'DISTSTYLE ALL',
        'rationale': (
            '3650行极小，ALL 分布开销接近零。'
            '每个节点都有完整的日期维度，过滤在本地完成。'
        )
    }
]

print("=== 练习 3 参考答案: Redshift Distribution Style ===")
print()
for item in reference_answer_3:
    print(f"Table: {item['table']} ({item['size']})")
    print(f"  Distribution: {item['distkey']}")
    print(f"  Rationale: {item['rationale']}")
    print()

print("\nFull DDL:")
full_ddl = """
CREATE TABLE fact_sales (
    sale_id     BIGINT, customer_id BIGINT, product_id INT, date INT, amount DECIMAL(12,2)
) DISTKEY(customer_id) COMPOUND SORTKEY(date, customer_id);

CREATE TABLE dim_customers (
    customer_id BIGINT, name VARCHAR(255), country VARCHAR(50), segment VARCHAR(50)
) DISTKEY(customer_id) SORTKEY(customer_id);

CREATE TABLE dim_products (
    product_id INT, name VARCHAR(255), category VARCHAR(100), price DECIMAL(10,2)
) DISTSTYLE ALL SORTKEY(product_id);

CREATE TABLE dim_date (
    date_id INT, date DATE, week INT, month INT, quarter INT, year INT
) DISTSTYLE ALL SORTKEY(date_id);
"""
print(full_ddl)

## 练习 4: Time Travel 恢复场景

**场景**: 数据工程师误执行了以下语句：
```sql
-- 错误: 本意是更新 2024 年的数据，但忘记 WHERE 子句
UPDATE orders SET status = 'CANCELLED';
-- Query ID: 019c1234-abcd-efgh-0000-0001aabbccdd
```

5 分钟后发现问题，orders 表有 50 亿行。

**问题**:
1. 写出使用 Snowflake Time Travel 的恢复 SQL
2. 这个操作有什么风险？如何降低风险？
3. 如果 Time Travel 窗口已过期（超过90天），还有什么选择？

In [ ]:
# 练习 4 参考答案

recovery_sql = """
-- ============================================================
-- 方法1: 用历史版本覆盖当前表 (适合表不是特别大时)
-- ============================================================

-- Step 1: 验证历史数据正确性
SELECT COUNT(*), status, COUNT(DISTINCT status)
FROM orders BEFORE (STATEMENT => '019c1234-abcd-efgh-0000-0001aabbccdd')
GROUP BY status
LIMIT 10;

-- Step 2: 先克隆当前（错误）状态作为存证
CREATE TABLE orders_bad_20240301 CLONE orders;

-- Step 3: 用历史版本覆盖（50亿行，时间较长）
CREATE OR REPLACE TABLE orders AS
    SELECT * FROM orders BEFORE (STATEMENT => '019c1234-abcd-efgh-0000-0001aabbccdd');

-- ============================================================
-- 方法2: 只恢复被错误修改的行 (更精准，减少写入量)
-- ============================================================

-- 用历史版本中的正确 status 更新当前表
MERGE INTO orders AS current
USING (
    SELECT order_id, status
    FROM orders BEFORE (STATEMENT => '019c1234-abcd-efgh-0000-0001aabbccdd')
) AS historical
ON current.order_id = historical.order_id
WHEN MATCHED AND current.status != historical.status THEN
    UPDATE SET current.status = historical.status;
"""

risks_and_mitigations = """
风险与缓解措施:

1. 风险: CREATE OR REPLACE TABLE 会产生大量计算和存储费用（50亿行重写）
   缓解: 优先使用 MERGE 方式，只更新变化的行

2. 风险: 恢复过程中表不可用（会锁表）
   缓解: 先 CLONE 到新表名，验证后原子性切换
   CREATE TABLE orders_recovered CLONE orders BEFORE (...);
   ALTER TABLE orders RENAME TO orders_bad_backup;
   ALTER TABLE orders_recovered RENAME TO orders;

3. 风险: 恢复操作本身出错
   缓解: 已在 Step 2 克隆了当前状态，可以再次恢复

4. 预防措施:
   - 为 UPDATE/DELETE 语句添加 WHERE 子句检查机制（代码审查）
   - 使用 Snowflake Tasks 在生产环境强制 DRY RUN
   - 重要表设置 DATA_RETENTION_TIME_IN_DAYS = 30（Enterprise）
"""

failsafe_options = """
Time Travel 过期后的选项:

1. Fail-Safe 期 (7天): 联系 Snowflake 支持，他们可以从 Fail-Safe 中恢复
   → 费用高，且不保证一定成功

2. 外部备份: 如果有定期 COPY INTO 到 S3/Azure Blob 的数据备份
   COPY INTO orders FROM @my_stage/backup/orders/2024-03-01/;

3. 上游重跑: 从数据源或 CDC 日志重新加载
   → 需要数据源支持历史数据重放

4. 部分恢复: 从其他下游系统（报表快照、数据集市）推断正确值
"""

print("=== 恢复 SQL ===")
print(recovery_sql)
print("\n=== 风险与缓解 ===")
print(risks_and_mitigations)
print("\n=== Time Travel 过期后的选项 ===")
print(failsafe_options)

## 练习 5: 综合场景 — 云数仓选型

**场景**: 一家电商公司的数据栈现状：
- 数据量：原始事件数据 200TB，每天增加 1TB
- 工作负载：70% BI 报表（Tableau），20% ETL，10% 数据科学
- 云环境：纯 AWS
- 当前痛点：BI 查询并发高，ETL 影响报表查询性能；成本不可预测
- 预算：希望固定月费用

**问题**: 推荐哪个云数仓平台？需要设计哪些关键配置？写出你的推荐理由和配置方案。

In [ ]:
# 练习 5 参考答案

recommendation = """
推荐: Snowflake (部署在 AWS)

理由:
1. 工作负载隔离: Snowflake 的多 VW 设计天然解决 ETL 影响 BI 的问题
   - ETL VW 和 BI VW 完全独立，互不影响
   - BigQuery Reservation 虽然也能隔离，但在 AWS 环境部署 GCP 服务增加复杂性

2. 固定成本: 使用 Snowflake Credits 预购 + Auto-Suspend 可控制费用
   - BigQuery On-Demand 成本随扫描量波动（200TB表全扫描=$1000/次）
   - Snowflake 按 VW 运行时长计费，预算可预测

3. 高并发 BI: BI Multi-Cluster Warehouse 自动处理 Tableau 并发峰值

4. AWS 原生: Snowflake 数据存储在客户自己的 S3，无需跨云

配置方案:
"""

config_design = [
    {
        'warehouse': 'etl_wh',
        'size': 'XL',
        'min_cluster': 1,
        'max_cluster': 1,
        'auto_suspend': 60,
        'schedule': '每天凌晨 1-5 点运行',
        'monthly_credits_est': '16 credits/hr * 4hrs * 22days = 1408 credits'
    },
    {
        'warehouse': 'bi_wh',
        'size': 'M',
        'min_cluster': 1,
        'max_cluster': 4,
        'auto_suspend': 300,
        'schedule': '工作时间 9am-7pm 活跃',
        'monthly_credits_est': '平均2集群 * 4cr/hr * 10hrs * 22days = 1760 credits'
    },
    {
        'warehouse': 'ds_wh',
        'size': 'L',
        'min_cluster': 1,
        'max_cluster': 1,
        'auto_suspend': 120,
        'schedule': '按需使用，估计每天2小时',
        'monthly_credits_est': '8 credits/hr * 2hrs * 22days = 352 credits'
    }
]

print(recommendation)

total_credits = 0
for wh in config_design:
    print(f"\nWarehouse: {wh['warehouse']}")
    for k, v in wh.items():
        if k != 'warehouse':
            print(f"  {k}: {v}")

print("""
其他关键配置:
1. Clustering Key: 事件表按 (region, DATE_TRUNC('day', event_ts)) 聚簇
2. Data Retention: 重要业务表设置 30 天 Time Travel
3. Zero-Copy Clone: 数据科学团队从生产数据克隆测试环境
4. Resource Monitor: 设置每月 Credit 预算告警（80%/100% 阈值）
5. Column Level Security: BI 用户屏蔽 PII 字段（Dynamic Data Masking）
""")

---
# 复习要点

## Snowflake

- **Virtual Warehouse**: 计算与存储分离的核心。纵向扩展（Size 升级）= 加速单查询；横向扩展（Multi-Cluster）= 解决并发排队
- **Auto-Suspend/Resume**: 按秒计费（最少60秒），是 Snowflake 成本控制的核心机制
- **Clustering Key**: 用于超大表减少微分区扫描。选列原则：(低基数列, 高频过滤列)，按选择性从低到高排列
- **Time Travel**: Enterprise 版支持最长 90 天历史查询/恢复，过期后有 7 天 Fail-Safe（仅 Snowflake 可操作）
- **Zero-Copy Clone**: 瞬时元数据克隆，Copy-on-Write 语义，写入分叉后才产生额外存储费用

## BigQuery

- **计费模型**: On-Demand（$5/TB 扫描）vs Reservation（按 Slot 固定费用）；Slot = 计算资源单元
- **Partition**: 物理分隔数据 → 直接减少计费字节；支持时间列/摄入时间/整数范围
- **Cluster**: 分区内排序 → 加速查询延迟（不直接减费用）；最多4列，顺序决定剪枝效果
- **BI Engine**: 内存加速层，为 Looker/Data Studio 等 BI 工具提供亚秒响应
- **`require_partition_filter = TRUE`**: 大表防护，强制分区过滤防止意外全表扫描

## Redshift

- **DISTSTYLE KEY**: 大事实表，按 JOIN 键分布，实现 Collocated JOIN（零数据移动）
- **DISTSTYLE ALL**: 小维度表（< 数百万行），每节点完整副本，JOIN 本地化
- **DISTSTYLE EVEN**: 无 JOIN 或无明显 distkey 的表，均匀分布防止倾斜
- **Compound Sort Key**: 有主要前缀过滤列时使用（效率高，维护成本低）
- **VACUUM + ANALYZE**: 定期维护命令，回收空间 + 更新统计信息

## Databricks

- **Photon**: C++ 向量化执行引擎，利用 SIMD 批量处理列数据；SQL 工作负载 2-8x 加速；Python UDF 不受益
- **Delta Engine**: Data Skipping (列统计跳过文件) + Z-Order (多维排序) + Auto Optimize (小文件合并)
- **Liquid Clustering**: DBR 13.3+ 新特性，替代 Z-Order，支持增量重聚簇，无需 OPTIMIZE 全量重写
- **Delta Time Travel**: `versionAsOf` / `timestampAsOf`，默认保留 30 天历史；VACUUM 清理时注意 retention 设置

## 面试高频知识点

1. Snowflake 的 Multi-Cluster vs Resize 解决不同问题（并发 vs 单查询）
2. BigQuery Partition 减费用，Cluster 减延迟，两者可以叠加使用
3. Redshift Collocated JOIN = 两表用相同列作为 DISTKEY
4. Zero-Copy Clone 不复制数据，只复制元数据指针（面试常考原理）
5. Photon 不能加速 Python UDF，面试官喜欢考这个限制